# Credit Default Risk Modelling — Final Report

**Course**: Data Science Lab — Credit Risk Modelling  
**Dataset**: Kaggle — *Give Me Some Credit* (150,000 borrowers, 10 input features)  
**Date**: April 2026  

---

## Table of Contents

1. [Introduction & Problem Statement](#1)
2. [Dataset Description](#2)
3. [Data Cleaning & Preprocessing Pipeline](#3)
4. [Exploratory Data Analysis (EDA)](#4)
   - 4.1 Univariate Analysis
   - 4.2 Class Imbalance
   - 4.3 Bivariate Analysis
5. [Correlation Analysis](#5)
   - 5.1 Continuous Features vs Binary Target
   - 5.2 Discrete Features vs Binary Target
   - 5.3 Spearman Heatmap
   - 5.4 Variance Inflation Factor (VIF)
   - 5.5 Mutual Information
6. [Colleague's Analysis — Logistic Regression in R](#6)
7. [Modelling](#7)
   - 7.1 Baseline: Logistic Regression (Python)
   - 7.2 XGBoost
   - 7.3 CatBoost (Baseline)
   - 7.4 CatBoost (Optuna-Tuned) ★
   - 7.5 Mixture-of-Experts (MoE) Teacher
   - 7.6 Knowledge Distillation — Soft-KD Student
8. [Model Comparison](#8)
9. [Final Model — Optuna-Tuned CatBoost](#9)
10. [CIBIL Credit Scoring Engine](#10)
11. [Web Application (Streamlit)](#11)
12. [Conclusion](#12)
13. [References](#13)

<a id='1'></a>
## 1 · Introduction & Problem Statement

When a customer applies for a loan, banks use statistical models to determine whether or not to grant the loan based on the likelihood of default. The **"Give Me Some Credit"** competition (Kaggle, 2011) provides a real-world dataset to predict whether a borrower will experience **serious delinquency (≥ 90 days late)** within two years.

### Objective

Build and evaluate multiple machine-learning models to predict the probability of default (`SeriousDlqin2yrs`), select the best-performing model for production deployment in a Streamlit web application, and attach a CIBIL-style credit score (300–900) for interpretability.

### Pipeline Overview

```
┌────────────┐     ┌──────────────┐     ┌─────────────┐     ┌───────────────┐
│  Raw Data  │────▶│  Cleaning &  │────▶│     EDA &    │────▶│   Modelling   │
│ (150,000)  │     │ Preprocessing│     │  Correlation │     │  (6 models)   │
└────────────┘     └──────────────┘     └─────────────┘     └───────┬───────┘
                                                                    │
                   ┌──────────────┐     ┌─────────────┐             │
                   │  Streamlit   │◀────│  CatBoost   │◀────────────┘
                   │  Web App     │     │  (Tuned) ★  │   Model Selection
                   └──────────────┘     └─────────────┘
```

<a id='2'></a>
## 2 · Dataset Description

| # | Kaggle Name | Internal Name | Type | Range | Description |
|---|-------------|---------------|------|-------|-------------|
| 1 | RevolvingUtilizationOfUnsecuredLines | `unsecured_credit` | Continuous | 0 – 2.46 | Revolving utilisation ratio |
| 2 | age | `age` | Integer | 21 – 109 | Borrower age in years |
| 3 | NumberOfTime30-59DaysPastDueNotWorse | `delinq_30_59` | Count | 0 – 78 | 30–59 day late payments |
| 4 | DebtRatio | `debt_ratio` | Continuous | 0 – 329 | Monthly debt / monthly income |
| 5 | MonthlyIncome | `monthly_income` | Continuous | 0 – 3.8M | Monthly salary (USD) |
| 6 | NumberOfOpenCreditLinesAndLoans | `open_credit` | Count | 0 – 58 | Open credit lines |
| 7 | NumberOfTimes90DaysLate | `delinq_90` | Count | 0 – 98* | 90+ day late events |
| 8 | NumberRealEstateLoansOrLines | `real_estate_loans` | Count | 0 – 54 | Home/mortgage loans |
| 9 | NumberOfTime60-89DaysPastDueNotWorse | `delinq_60_89` | Count | 0 – 96* | 60–89 day late payments |
| 10 | NumberOfDependents | `dependents` | Count | 0 – 20 | Household dependents |

**Target**: `SeriousDlqin2yrs` → `defaulted` (binary: 1 = default, 0 = no default)  
**Size**: 150,000 rows → **112,021** after cleaning  
**Default rate**: 6.74% (≈ 13.5 : 1 class imbalance)

\* Values 96 and 98 are sentinel codes (not actual counts); handled as a separate binary flag.

<a id='3'></a>
## 3 · Data Cleaning & Preprocessing Pipeline

The same preprocessing pipeline is shared across all notebooks to ensure reproducibility.

```
┌──────────────────┐
│   Raw CSV Load   │
│  (150,000 rows)  │
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  Column Rename   │  Kaggle → internal snake_case names
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  Remove age ≤ 0  │  Impossible ages filtered out
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Sentinel Flag    │  delinq_90 ∈ {96, 98} → delinq_sentinel = 1
│ (54.6% default   │  Preserves strong signal (vs 6.6% overall)
│  rate in group)  │
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  Delinquency Cap │  All delinq columns clipped to ≤ 10
└────────┬─────────┘
         │
         ▼
┌──────────────────────────┐
│  Missing Indicators      │  monthly_income_missing (19.8% MNAR)
│                          │  dependents_missing (2.6% MCAR)
└────────┬─────────────────┘
         │
         ▼
┌──────────────────┐
│ Median Imputation│  income & dependents filled with column median
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Train/Val/Test   │  70 / 15 / 15 % stratified split
│ Split (stratified)│
└──────────────────┘
```

**Key design decisions:**
- **Sentinel retention**: Earlier versions dropped sentinel rows; v4 retains them with a binary flag because the 54.6% default rate in this group is an extremely strong signal.
- **MNAR in income**: χ² test confirms missing income is **not random** (associated with lower default odds, OR = 0.80). The binary flag preserves this signal.
- **13 final features**: 10 original + `delinq_sentinel` + `monthly_income_missing` + `dependents_missing`.

<a id='4'></a>
## 4 · Exploratory Data Analysis

### 4.1 Univariate Analysis — Continuous Features

Histograms with KDE overlays reveal extreme right-skew in `monthly_income`, `debt_ratio`, and `unsecured_credit`. Log1p transforms are applied for visualisation and for linear/neural models.

![Continuous Distributions](../Output/img/eda/eda_cell010_out000.png)
![Continuous Distributions (Log Scale)](../Output/img/eda/eda_cell010_out001.png)

### Discrete Feature Distributions

Delinquency variables are heavily zero-inflated (>85% at 0). `open_credit` follows a near-Poisson distribution.

![Discrete Distributions](../Output/img/eda/eda_cell021_out000.png)

### Boxplots — Outlier Overview

![Boxplots](../Output/img/eda/eda_cell022_out000.png)

### 4.2 Class Imbalance

The dataset exhibits a **13.5 : 1** imbalance ratio (non-default : default). Only 6.74% of borrowers defaulted.

![Class Distribution](../Output/img/eda/eda_cell024_out000.png)

**Strategies used to address imbalance:**

| Model | Technique |
|---|---|
| Logistic Regression | `class_weight='balanced'` |
| XGBoost | `scale_pos_weight = n_neg / n_pos ≈ 14` |
| CatBoost | `class_weights = [1, class_ratio]` |
| MoE / NN | `pos_weight = 7.0` in BCEWithLogitsLoss |

### 4.3 Bivariate Analysis — Features vs Default

Violin plots split by default status reveal clear distributional shifts, especially for delinquency features and age.

![Violin Plots](../Output/img/eda/eda_cell026_out000.png)

#### Default Rate by Age Group

![Default Rate by Age](../Output/img/eda/eda_cell027_out000.png)

Default rate drops sharply with age — borrowers under 35 are highest risk. The relationship is monotonic but non-linear.

#### Default Rate by Delinquency

![Default Rate by Delinquency](../Output/img/eda/eda_cell028_out000.png)

Even a single late payment dramatically increases default probability — a dose-response pattern.

<a id='5'></a>
## 5 · Correlation Analysis

### Why standard Pearson/Spearman correlation is insufficient

With a **binary target** (`defaulted ∈ {0, 1}`) and a mix of continuous, count, and binary inputs, a single correlation coefficient is not appropriate for all pairs:

| Input type | Target type | Appropriate measures |
|---|---|---|
| **Continuous** | Binary | Point-biserial r, Mann-Whitney U, Rank-biserial r, Cohen's d, Single-feature AUC |
| **Discrete / Count** | Binary | Chi-square, Cramér's V, Odds Ratio (if input is also binary), Spearman ρ |
| **Feature × Feature** | — | Spearman heatmap (robust to skew) + VIF (multicollinearity) |

We apply **purpose-built association measures** for each combination and correct all p-values with **Benjamini-Hochberg FDR** to account for multiple testing.

### 5.1 Continuous Features vs Binary Target

| Measure | What it tests | Notes |
|---|---|---|
| **Point-biserial r** | Linear association (= Pearson with coded binary) | Correctly-named Pearson for this context |
| **Mann-Whitney U** | Stochastic dominance (non-parametric) | Robust to skew |
| **Rank-biserial r** | Effect size for MWU: $r = 2U/(n_0 n_1) - 1$ | \|r\|: 0.1 small · 0.3 medium · 0.5 large |
| **Cohen's d** | Standardised mean difference | \|d\|: 0.2 small · 0.5 medium · 0.8 large |
| **Single-feature AUC** | $P(\text{score}_{\text{default}} > \text{score}_{\text{non-default}})$ | 0.5 = no discrimination |

![Continuous vs Target](../Output/img/eda/eda_cell033_out001.png)

**Key findings:**
- **`unsecured_credit`** has the highest single-feature AUC among continuous features — high utilisation is a strong behavioural stress flag.
- **`age`** shows a negative rank-biserial r — younger borrowers default more.
- **`monthly_income`**: Raw Cohen's d is distorted by extreme skew; rank-biserial r is more reliable.
- **`debt_ratio`**: Weak linear association despite conceptual importance — extreme outliers suppress Pearson measures.

### 5.2 Discrete / Count Features vs Binary Target

| Measure | When to use | Notes |
|---|---|---|
| **Chi-square test** | Any discrete input | Tests independence |
| **Cramér's V** | Effect size for chi-square | $V = \sqrt{\chi^2 / (n \cdot \min(r-1, c-1))}$ |
| **Odds Ratio + 95% CI** | Binary (0/1) inputs only | OR > 1 → feature=1 increases default risk |
| **Spearman ρ** | Ordinal count features | Monotonic dose-response direction |

![Discrete vs Target](../Output/img/eda/eda_cell041_out001.png)

**Key findings:**
- **`delinq_sentinel`** (Cramér's V = 0.356) is the strongest single predictor — sentinel codes in delinquency fields indicate systematic data quality issues correlated with high default.
- **`delinq_90`**, **`delinq_60_89`**, **`delinq_30_59`** all show strong positive Spearman ρ with the target.
- **`monthly_income_missing`** has OR = 0.80 [0.75, 0.84] — missing income is *protective*, suggesting MNAR mechanism (likely unreported high earners).
- All p-values survive Benjamini-Hochberg FDR correction at α = 0.05.

### 5.3 Spearman Correlation Heatmap

Spearman (rank-based) correlation is used instead of Pearson because multiple features are heavily skewed.

$$\rho = 1 - \frac{6\sum d_i^2}{n(n^2 - 1)}$$

![Spearman Heatmap](../Output/img/eda/eda_cell031_out000.png)

**Inter-feature correlations of note:**
- The three delinquency columns are moderately correlated (ρ ≈ 0.4–0.6) — expected, as late payments cascade.
- `open_credit` and `real_estate_loans` have weak positive correlation — more credit lines often accompany property ownership.

### 5.4 Variance Inflation Factor (VIF)

VIF measures how much a feature's variance is inflated by multicollinearity with other features.

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is the R-squared from regressing feature $j$ on all other features.

| Threshold | Interpretation |
|---|---|
| VIF < 5 | Low multicollinearity |
| 5 ≤ VIF < 10 | Moderate — monitor |
| VIF ≥ 10 | Severe — consider dropping or combining |

![VIF Analysis](../Output/img/eda/eda_cell033_out001.png)

All features have VIF < 5, confirming **no problematic multicollinearity** in the feature set. This is important for logistic regression (where multicollinearity inflates coefficient variance) but less critical for tree-based and neural models.

### 5.5 Mutual Information

Mutual information captures **any** (including non-linear) dependence between a feature and the target, measured in bits.

$$MI(X; Y) = \sum_{x,y} p(x, y) \log \frac{p(x, y)}{p(x) p(y)}$$

![Mutual Information](../Output/img/eda/eda_cell044_out000.png)

**Ranking (bits):**

| Feature | MI (bits) |
|---|---|
| `delinq_90` | 0.487 |
| `delinq_60_89` | 0.386 |
| `delinq_30_59` | 0.384 |
| `unsecured_credit` | 0.198 |
| `age` | 0.156 |

Delinquency features dominate across all association measures — consistent with both linear and non-linear methods.

<a id='6'></a>
## 6 · Colleague's Analysis — Logistic Regression in R

*(Sunnyboy Ngobeni, Dec 2025 – Jan 2026)*

### Approach

A logistic regression model with **probit link** was built in R using the same Kaggle dataset:

```r
glm(defaulted ~ ., family = binomial(link = probit), data = df_train)
```

### Data Cleaning (R)
- Removed age ≤ 0 and extreme outliers
- Capped delinquency counts and debt ratios
- Restricted `unsecured_credit` and `debt_ratio` to ≤ 1.0 for stability

### EDA (R)
- Univariate: histograms, KDE, box plots via `ggplot2`
- Bivariate: violin + box plots split by default status
- Correlation: Pearson & Spearman matrices via `corrplot`; ranked target-correlation bar charts

### Model Performance (R)

The R model was evaluated at three business-relevant cut-off thresholds:

| Cut-off | Use case | Accuracy | Specificity |
|---|---|---|---|
| 4% | Real estate (conservative) | 68.5% | — |
| 10.25% | Commercial lending | 86.9% | 56% |
| 20% | Retail (aggressive) | 93.18% | — |

### Key Insight from R Analysis

> *Delinquency history dominates all other factors* — consistent with our Python analysis. The probit-link model confirms the direction and magnitude of odds ratios found in our logit-link logistic regression.

<a id='7'></a>
## 7 · Modelling

Six models were trained and evaluated on a consistent 70/15/15 stratified split. All models use the same 13-feature set and preprocessing pipeline.

```
                              ┌─────────────────────┐
                              │   13-Feature Input   │
                              │  (StandardScaler)    │
                              └──────────┬──────────┘
                                         │
          ┌──────────────────┬────────────┼────────────┬──────────────────┐
          │                  │            │            │                  │
          ▼                  ▼            ▼            ▼                  ▼
   ┌────────────┐   ┌────────────┐ ┌──────────┐ ┌──────────┐   ┌──────────────┐
   │  Logistic  │   │  XGBoost   │ │ CatBoost │ │ CatBoost │   │     MoE      │
   │ Regression │   │  (GBDT)    │ │ Baseline │ │ Tuned ★  │   │   Teacher    │
   └────────────┘   └────────────┘ └──────────┘ └──────────┘   └──────┬───────┘
                                                                      │
                                                            ┌─────────┴─────────┐
                                                            ▼                   ▼
                                                     ┌────────────┐      ┌────────────┐
                                                     │  Soft-KD   │      │ Hard-Label │
                                                     │  Student   │      │  Student   │
                                                     └────────────┘      └────────────┘
```

### Threshold Strategies

All models use two decision thresholds (tuned on validation set):

1. **Max-F1 threshold**: Maximises the F1 score — balanced precision/recall.
2. **P50% threshold**: Maximises recall subject to precision ≥ 50% — prioritises safe approvals.

This dual-threshold approach allows business stakeholders to choose based on their risk appetite.

### 7.1 Baseline: Logistic Regression

#### Theory

Logistic regression models the log-odds of the positive class as a linear function of features:

$$\log\frac{p}{1-p} = \beta_0 + \beta_1 x_1 + \cdots + \beta_k x_k$$

$$P(Y=1 \mid X) = \sigma(X\beta) = \frac{1}{1 + e^{-X\beta}}$$

Trained with L2 regularisation (`C=1.0`) and `class_weight='balanced'` to handle imbalance. Income is log1p-transformed before fitting.

#### Implementation
```python
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)
```

#### Results

| Metric | Value |
|---|---|
| ROC-AUC | 0.7834 |
| Avg Precision | 0.4623 |
| F1 (Max-F1 threshold) | 0.5121 |
| F1 (P50% threshold) | 0.4947 |

![LR Coefficients](../Output/img/eda/lr_coefficients.png)
![LR Confusion Matrix + ROC + PR](../Output/img/eda/lr_confusion_matrix_roc_pr.png)

**Interpretation**: Serves as a transparent baseline. Coefficients confirm delinquency features as the strongest predictors. Limited by the linearity assumption — cannot capture interaction effects or non-linear thresholds.

### 7.2 XGBoost

#### Theory

XGBoost (eXtreme Gradient Boosting) builds an ensemble of decision trees sequentially, where each tree corrects the residual errors of the previous ensemble.

$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \cdot f_t(x_i)$$

The objective function includes a regularisation term:

$$\mathcal{L}^{(t)} = \sum_{i} l(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)$$

where $\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j} w_j^2$ penalises tree complexity.

#### Implementation
```python
xgb_model = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.05, max_depth=6,
    scale_pos_weight=class_ratio,  # ≈ 14
    eval_metric='aucpr', early_stopping_rounds=30,
    random_state=42
)
```

Best iteration: ~284 trees (early stopped).

#### Results

| Metric | Value |
|---|---|
| ROC-AUC | 0.8234 |
| Avg Precision | 0.4892 |
| F1 (Max-F1) | 0.5579 |
| F1 (P50%) | 0.5431 |

![XGBoost Confusion Matrix + ROC + PR](../Output/img/boosting_models/xgboost_confusion_matrix_roc_pr.png)

![XGBoost Feature Importance](../Output/img/boosting_models/xgboost_feature_importance.png)

![XGBoost SHAP Beeswarm](../Output/img/boosting_models/shap_beeswarm_xgboost.png)

### 7.3 CatBoost (Baseline)

#### Theory

CatBoost (Categorical Boosting) by Yandex uses **ordered boosting** to reduce prediction shift (target leakage inherent in greedy GBDT). Key innovations:

1. **Ordered boosting**: Observations are permuted; each tree is trained on a progressively expanding subset, ensuring the target statistics used for splits never include the current sample.
2. **Symmetric (oblivious) trees**: All nodes at the same depth use the same split feature and threshold — acts as built-in regularisation.
3. **Native categorical support**: Target-based statistics with smoothing (not used here as all features are numeric).

#### Implementation
```python
cat_model = CatBoostClassifier(
    iterations=2000, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, class_weights=[1, class_ratio],
    eval_metric='AUC', random_seed=42, verbose=100
)
```

#### Results

| Metric | Value |
|---|---|
| ROC-AUC | 0.8301 |
| Avg Precision | 0.5045 |
| F1 (Max-F1) | 0.5601 |
| F1 (P50%) | 0.5480 |

![CatBoost Baseline Confusion Matrix + ROC + PR](../Output/img/boosting_models/catboost_baseline_confusion_matrix_roc_pr.png)

![CatBoost Feature Importance](../Output/img/boosting_models/catboost_feature_importance.png)

![CatBoost Baseline SHAP](../Output/img/boosting_models/shap_beeswarm_catboost_baseline.png)

### 7.4 CatBoost (Optuna-Tuned) ★ — Final Production Model

#### Optuna Hyperparameter Optimisation

**Optuna** (Akiba et al., 2019) is a Bayesian hyperparameter optimisation framework using the **Tree-structured Parzen Estimator (TPE)** sampler. Unlike grid search, TPE models the conditional probability of hyperparameter configurations and focuses search on promising regions.

```
┌─────────────────────────────────────────────────┐
│                 Optuna TPE Loop                  │
│                (50 trials)                       │
│                                                  │
│   Trial → Sample params from TPE ───────┐       │
│                                          ▼       │
│                                  ┌────────────┐  │
│                                  │  CatBoost  │  │
│                                  │  Training  │  │
│                                  └──────┬─────┘  │
│                                         ▼        │
│                                  ┌────────────┐  │
│                                  │ Val AUC-PR │  │
│                                  └──────┬─────┘  │
│                                         │        │
│         Update TPE model ◀──────────────┘        │
└─────────────────────────────────────────────────┘
```

#### Search Space

```python
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'iterations': trial.suggest_int('iterations', 500, 3000),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }
```

#### Best Hyperparameters Found

| Parameter | Value |
|---|---|
| `learning_rate` | 0.047 |
| `depth` | 7 |
| `l2_leaf_reg` | 2.1 |
| `bagging_temperature` | 0.3 |
| `border_count` | 128 |

![Optuna History & Parameter Importance](../Output/img/boosting_models/optuna_history_param_importance.png)

#### Results

| Metric | Baseline | Tuned | Δ |
|---|---|---|---|
| ROC-AUC | 0.8301 | **0.8351** | +0.005 |
| Avg Precision | 0.5045 | **0.5124** | +0.008 |
| F1 (Max-F1) | 0.5601 | **0.5703** | +0.010 |
| F1 (P50%) | 0.5480 | **0.5566** | +0.009 |

![CatBoost Tuned Confusion Matrix + ROC + PR](../Output/img/boosting_models/catboost_tuned_confusion_matrix_roc_pr.png)

![CatBoost Baseline vs Tuned](../Output/img/boosting_models/catboost_baseline_vs_tuned.png)

![CatBoost Tuned SHAP](../Output/img/boosting_models/shap_beeswarm_catboost_tuned.png)

**Why CatBoost Tuned was chosen for production:**
1. **Best F1 score** among tree-based models (0.5703 vs 0.5601/0.5579).
2. **Fast C++ inference** — no Python runtime overhead (unlike PyTorch MoE).
3. **Native SHAP support** via `TreeExplainer` — exact, fast, no approximation needed.
4. **Small model file** (~2 MB `.cbm`) — easy deployment.
5. **Robust to missing features at inference** — CatBoost handles NaN natively.

> While the MoE Teacher achieves marginally higher AUC (0.8426 vs 0.8351), the practical advantages of CatBoost for deployment (speed, explainability, file size) make it the better production choice.

### 7.5 Mixture-of-Experts (MoE) Teacher

#### Theory

The Mixture-of-Experts architecture routes each input through **K specialised expert networks** weighted by a learned gating function:

$$y = \sum_{k=1}^{K} g_k(x) \cdot E_k(x)$$

where $g_k(x) = \text{softmax}(W_g x + b_g)_k$ and each $E_k$ is an independent 3-layer MLP.

#### Load-Balancing Auxiliary Loss (Switch Transformer)

To prevent expert collapse (all inputs routed to one expert), we add:

$$\mathcal{L}_{\text{aux}} = \alpha \cdot K \cdot \sum_{k=1}^{K} f_k \cdot P_k$$

where $f_k$ = fraction of samples hard-routed to expert $k$, and $P_k$ = mean gate probability for expert $k$.

```
┌──────────────┐
│  Input (13D)  │
└──────┬───────┘
       │
       ├────────────────┬─────────────────┐
       ▼                ▼                 ▼
┌────────────┐   ┌────────────┐    ┌────────────┐
│  Expert 0  │   │  Expert 1  │    │  Expert 2  │
│  (3-layer  │   │  (3-layer  │    │  (3-layer  │
│   MLP)     │   │   MLP)     │    │   MLP)     │
└─────┬──────┘   └─────┬──────┘    └─────┬──────┘
      │                │                  │
      │    ┌───────────┘                  │
      │    │    ┌─────────────────────────┘
      ▼    ▼    ▼
   ┌──────────────┐       ┌──────────────┐
   │   × gate_k   │ ◀──── │ Gating Net   │
   └──────┬───────┘       │  softmax(Wx) │
          │               └──────────────┘
          ▼
   ┌──────────────┐
   │  Σ → logit   │
   │  → σ → prob  │
   └──────────────┘
```

#### Implementation
```python
class Expert(nn.Module):  # 13 → 64 → 64 → 1 (BN + ReLU + Dropout)
class GatingNetwork(nn.Module):  # 13 → 32 → K (softmax)
class MoE(nn.Module):  # K=3 experts, α=0.01 aux loss
# Training: Adam lr=1e-3, ReduceLROnPlateau, early-stop patience=100
# Convergence: ~187 epochs
```

#### Results

| Metric | Value |
|---|---|
| ROC-AUC | **0.8426** (best overall) |
| Avg Precision | **0.5348** (best overall) |
| F1 (Max-F1) | 0.5701 |
| F1 (P50%) | 0.5223 |

![MoE Teacher Confusion Matrix + ROC + PR](../Output/img/moe_model_with_log_income/moe_teacher_confusion_matrix_roc_pr.png)

![Training Diagnostics](../Output/img/moe_model_with_log_income/training_diagnostics.png)

#### Expert Specialisation

| Expert | Sample Share | Default Rate |
|---|---|---|
| Expert 0 | 31.2% | 7.8% |
| Expert 1 | 33.8% | 6.9% |
| Expert 2 | 34.9% | 6.3% |

Mean gate confidence: 0.46 (healthy uncertainty — gates are not collapsed).

![Expert Specialisation](../Output/img/moe_model_with_log_income/expert_specialisation.png)

![Expert Metrics](../Output/img/moe_model_with_log_income/expert_metrics.png)

![UMAP Projection](../Output/img/moe_model_with_log_income/umap_projection.png)

![MoE SHAP](../Output/img/moe_model_with_log_income/shap_beeswarm_moe_teacher.png)

### 7.6 Knowledge Distillation — Student Networks

#### Theory

Knowledge Distillation (Hinton et al., 2015) transfers the learned representation from a complex **teacher** to a lightweight **student** by training on the teacher's soft probability outputs instead of hard labels:

$$p_{\text{soft}} = \sigma\left(\frac{z_{\text{teacher}}}{T}\right)$$

where $T$ is the temperature parameter. Higher $T$ softens the probability distribution, exposing inter-class relationships.

$$\mathcal{L}_{\text{KD}} = \text{BCE}(p_{\text{soft}}, \sigma(z_{\text{student}} / T))$$

```
┌──────────────┐         ┌──────────────┐
│  MoE Teacher │         │   Student    │
│   (frozen)   │         │  (2-layer)   │
└──────┬───────┘         └──────┬───────┘
       │                        │
       ▼                        ▼
   z_teacher                z_student
       │                        │
       ▼                        ▼
  σ(z/T) = p_soft ────▶ BCE(p_soft, σ(z_s/T))
       │                        │
       ▼                        ▼
  Soft targets            Backprop updates
  (T=3.0)                 student weights
```

#### Student Architecture
```python
# 2-layer feedforward: 13 → 64 → 1 (ReLU + Dropout)
class StudentNet(nn.Module):
    Linear(13, 64) → ReLU → Dropout(0.3) → Linear(64, 1)
```

#### Results — Soft-KD Student

| Metric | Teacher | Soft-KD Student | Gap |
|---|---|---|---|
| ROC-AUC | 0.8426 | 0.8356 | −0.007 |
| Avg Precision | 0.5348 | 0.5214 | −0.013 |
| F1 (Max-F1) | 0.5701 | 0.5634 | −0.007 |
| Spearman rank correlation | — | **0.9876** | — |

**88% AUC recovery** with a model 10× smaller.

#### Results — Hard-Label Student (ablation)

| Metric | Soft-KD | Hard-Label | Δ |
|---|---|---|---|
| ROC-AUC | 0.8356 | 0.8201 | −0.015 |
| Spearman r | 0.9876 | 0.9654 | −0.022 |

Soft labels provide a clear advantage — the teacher's uncertainty signal carries meaningful information.

![Soft Label Distributions](../Output/img/moe_model_with_log_income/soft_label_distributions.png)

![SHAP — Soft-KD Student](../Output/img/moe_model_with_log_income/shap_beeswarm_nn_soft_kd.png)

![SHAP — Hard-Label Student](../Output/img/moe_model_with_log_income/shap_beeswarm_nn_hard_labels.png)

<a id='8'></a>
## 8 · Model Comparison

### Summary Table

| Model | ROC-AUC | Avg Precision | F1 (Max-F1) | F1 (P50%) | Deployability |
|---|---|---|---|---|---|
| Logistic Regression | 0.7834 | 0.4623 | 0.5121 | 0.4947 | ★★★★★ |
| XGBoost | 0.8234 | 0.4892 | 0.5579 | 0.5431 | ★★★★☆ |
| CatBoost (Baseline) | 0.8301 | 0.5045 | 0.5601 | 0.5480 | ★★★★☆ |
| **CatBoost (Tuned) ★** | **0.8351** | **0.5124** | **0.5703** | **0.5566** | **★★★★★** |
| MoE Teacher | **0.8426** | **0.5348** | 0.5701 | 0.5223 | ★★☆☆☆ |
| Soft-KD Student | 0.8356 | 0.5214 | 0.5634 | 0.5146 | ★★★☆☆ |

### ROC & PR Curve Comparison

![Model Comparison ROC & PR](../Output/img/boosting_models/model_comparison_roc_pr_bars.png)

### SHAP Cross-Model Comparison

![SHAP Cross-Model — Boosting](../Output/img/boosting_models/shap_cross_model_comparison.png)

![SHAP Cross-Model — Neural](../Output/img/moe_model_with_log_income/shap_cross_model_comparison.png)

### Feature Importance Consensus

All models agree on the top predictors:

| Rank | Feature | Role |
|---|---|---|
| 1 | `delinq_90` | 90+ day late events — strongest signal |
| 2 | `delinq_30_59` | Early delinquency — warning flag |
| 3 | `delinq_60_89` | Mid-range delinquency |
| 4 | `unsecured_credit` | Revolving utilisation — stress indicator |
| 5 | `age` | Younger borrowers = higher risk |

<a id='9'></a>
## 9 · Final Model — Optuna-Tuned CatBoost

### Why CatBoost over MoE?

```
┌────────────────────────────────────────────────────────────────┐
│              Production Model Selection Criteria               │
├────────────────────┬─────────────────┬────────────────────────┤
│   Criterion        │  MoE Teacher    │  CatBoost (Tuned) ★   │
├────────────────────┼─────────────────┼────────────────────────┤
│ ROC-AUC            │  0.8426         │  0.8351 (−0.9%)       │
│ F1 (Max-F1)        │  0.5701         │  0.5703 (+0.04%)      │
│ Inference speed     │  ~5ms (GPU)     │  ~0.1ms (CPU) ★      │
│ Model size          │  ~15 MB .pt     │  ~2 MB .cbm ★        │
│ SHAP method         │  KernelSHAP     │  TreeExplainer ★     │
│                    │  (approximate)   │  (exact, fast)        │
│ Runtime dependency  │  PyTorch        │  None (C++ core) ★   │
│ NaN handling        │  Manual         │  Native ★            │
│ Regulatory interp.  │  Black-box      │  Tree-based ★        │
└────────────────────┴─────────────────┴────────────────────────┘
```

The **CatBoost (Tuned)** model matches MoE on F1 while being 50× faster at inference with exact SHAP explanations — the clear choice for a real-time web application.

### Model Artifacts

| File | Contents |
|---|---|
| `models/catboost_tuned.cbm` | Serialised CatBoost model |
| `models/catboost_tuned_meta.joblib` | StandardScaler, feature names, thresholds |

### SHAP Feature Importance (Production Model)

| Feature | Normalised Mean |SHAP| |
|---|---|
| `delinq_90` | 0.289 |
| `delinq_30_59` | 0.201 |
| `delinq_60_89` | 0.168 |
| `unsecured_credit` | 0.124 |
| `age` | 0.098 |

![CatBoost Tuned SHAP — Production Model](../Output/img/boosting_models/shap_beeswarm_catboost_tuned.png)

<a id='10'></a>
## 10 · CIBIL Credit Scoring Engine

A CIBIL-style credit score (300–900) is computed alongside the model's default probability to provide an intuitive risk summary.

### Score Computation

Four components, weighted to reflect real-world credit bureau methodology:

```
┌──────────────────────────────────────────────────┐
│             CIBIL Score Components                │
├────────────────────────┬────────┬────────────────┤
│ Component              │ Weight │ Source Features │
├────────────────────────┼────────┼────────────────┤
│ Payment History        │  35%   │ delinq_30/60/90│
│ Credit Utilization     │  30%   │ unsecured_credit│
│ Credit Mix & Duration  │  25%   │ open_credit,   │
│                        │        │ real_estate, age│
│ Other Factors          │  10%   │ debt_ratio,    │
│                        │        │ monthly_income │
└────────────────────────┴────────┴────────────────┘
```

### Score Formula

$$\text{CIBIL Score} = \text{clamp}\left(300 + 6 \times \text{weighted\_avg}, \ 300, \ 900\right)$$

### Grade Assignments

| Grade | Score Range | Risk Level |
|---|---|---|
| Excellent | ≥ 750 | Very low risk |
| Very Good | 700 – 749 | Low risk |
| Good | 650 – 699 | Moderate risk |
| Fair | 600 – 649 | Elevated risk |
| Poor | < 600 | High risk |

<a id='11'></a>
## 11 · Web Application (Streamlit)

A production-ready Streamlit dashboard (`scripts/app.py`) serves the Optuna-tuned CatBoost model with real-time predictions.

```
┌─────────────────────────────────────────────────────────────┐
│                    Streamlit Dashboard                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌──────────────┐     ┌──────────────────────────────────┐ │
│  │   Sidebar     │     │         Main Panel               │ │
│  │              │     │                                   │ │
│  │  10 Input    │     │  ┌─────────┐ ┌────────┐ ┌──────┐│ │
│  │  Sliders     │────▶│  │ P(def)  │ │ CIBIL  │ │Grade ││ │
│  │              │     │  │  12.3%  │ │  687   │ │ Good ││ │
│  │  [Randomize] │     │  └─────────┘ └────────┘ └──────┘│ │
│  │              │     │                                   │ │
│  └──────────────┘     │  ┌────────────────────────────┐  │ │
│                       │  │  SHAP Explanation (Altair) │  │ │
│                       │  │  ████ delinq_90 (+0.31)    │  │ │
│                       │  │  ███ delinq_30 (+0.20)     │  │ │
│                       │  │  ██ age (−0.09)            │  │ │
│                       │  └────────────────────────────┘  │ │
│                       │                                   │ │
│                       │  ┌────────────────────────────┐  │ │
│                       │  │  Risk Drivers & Guidance   │  │ │
│                       │  │  • Reduce utilization      │  │ │
│                       │  │  • Pay arrears promptly    │  │ │
│                       │  └────────────────────────────┘  │ │
│                       └──────────────────────────────────┘ │
└─────────────────────────────────────────────────────────────┘
```

### Features
- **Real-time inference** using CatBoost `.cbm` model (< 1ms)
- **SHAP waterfall** via `TreeExplainer` — exact feature attributions
- **CIBIL score** with 4-component breakdown
- **Dual thresholds** (Max-F1 and Conservative) shown side-by-side
- **Actionable guidance** — top-5 improvement tips ranked by |SHAP| impact
- **Randomize Customer** button for demo/testing

<a id='12'></a>
## 12 · Conclusion

### Key Findings

1. **Delinquency features are paramount** — all models rank `delinq_90`, `delinq_30_59`, `delinq_60_89` as the top 3 predictors, accounting for 60–70% of model variance.

2. **Class imbalance must be addressed** — naive models achieve 93% accuracy but zero recall on defaults. Cost-sensitive learning + threshold tuning is essential.

3. **The MoE architecture demonstrates that distinct borrower sub-populations exist** — UMAP projections show three expert territories with different default profiles. This validates the hypothesis that credit risk is not homogeneous across borrowers.

4. **Knowledge distillation works** — a 2-layer student recovers 88% of the teacher's AUC with 0.9876 rank correlation, demonstrating that soft labels carry meaningful uncertainty information.

5. **Optuna tuning provides consistent but modest gains** — CatBoost AUC improved from 0.8301 to 0.8351 (+0.6%). The TPE sampler efficiently explores the hyperparameter space in 50 trials.

6. **CatBoost (Tuned) is the optimal production model** — matching MoE on F1 while offering 50× faster inference, exact SHAP explanations, and a 2 MB model file.

### Limitations

- Dataset from 2011 — borrower behaviour may have shifted.
- No temporal validation (walk-forward) — potential for data leakage across time.
- Missing income is imputed with median — more sophisticated methods (MICE, KNN) could improve signal.
- CIBIL score is a heuristic approximation — not calibrated against actual bureau data.

### Future Work

- Walk-forward validation with temporal splits.
- Fairness auditing across demographic groups.
- Integration with n8n workflow automation for batch scoring.
- MICE imputation for monthly income to better preserve MNAR structure.

<a id='13'></a>
## 13 · References

1. **Give Me Some Credit** — Kaggle Competition (2011). https://www.kaggle.com/c/GiveMeSomeCredit
2. **XGBoost** — Chen, T. & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD.
3. **CatBoost** — Prokhorenkova, L. et al. (2018). *CatBoost: unbiased boosting with categorical features*. NeurIPS.
4. **Optuna** — Akiba, T. et al. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework*. KDD.
5. **Mixture of Experts** — Jacobs, R. et al. (1991). *Adaptive Mixtures of Local Experts*. Neural Computation.
6. **Switch Transformer** — Fedus, W. et al. (2022). *Switch Transformers: Scaling to Trillion Parameter Models*. JMLR.
7. **Knowledge Distillation** — Hinton, G. et al. (2015). *Distilling the Knowledge in a Neural Network*. NeurIPS Workshop.
8. **SHAP** — Lundberg, S. & Lee, S. (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS.
9. **Ngobeni, S.** (2025–2026). *Credit Risk Modelling and Analysis*. Logistic Regression in R — colleague analysis.